In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print("PyTorch version:", torch.__version__)

GPU available: True
GPU name: Tesla T4
PyTorch version: 2.9.0+cu126


**Install dependencies & clone YOLOv7**

In [2]:
import os

# Clone YOLOv7
os.system("git clone https://github.com/WongKinYiu/yolov7")
os.chdir("/kaggle/working/yolov7")

# Install requirements
os.system("pip install -r requirements.txt")
os.system("pip install torchvision Pillow tqdm matplotlib seaborn")

print("YOLOv7 cloned and dependencies installed!")


Cloning into 'yolov7'...


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 78.2 MB/s eta 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


YOLOv7 cloned and dependencies installed!


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║   COMPLETE YOLOv7 + GAN TRAINING NOTEBOOK                   ║
# ║   Dataset: Candy Pick & Place                                ║
# ║   Run each cell in order                                     ║
# ╚══════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────────────────────────
# CELL 1 — Check GPU
# ─────────────────────────────────────────────────────────────────
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print("PyTorch version:", torch.__version__)

# ─────────────────────────────────────────────────────────────────
# CELL 2 — Install dependencies & clone YOLOv7
# ─────────────────────────────────────────────────────────────────
import os

# Clone YOLOv7
os.system("git clone https://github.com/WongKinYiu/yolov7")
os.chdir("/kaggle/working/yolov7")

# Install requirements
os.system("pip install -r requirements.txt")
os.system("pip install torchvision Pillow tqdm matplotlib seaborn")

print("YOLOv7 cloned and dependencies installed!")

# ─────────────────────────────────────────────────────────────────
# CELL 3 — Set up paths
# ─────────────────────────────────────────────────────────────────
import os

# ── CHANGE THIS to match your Kaggle dataset name ──
DATASET_PATH = "/kaggle/input/candy-dataset"   # adjust if named differently

# Verify dataset exists
images_path = os.path.join(DATASET_PATH, "images")
labels_path = os.path.join(DATASET_PATH, "labels")
classes_path = os.path.join(DATASET_PATH, "classes.txt")

print("Images folder exists:", os.path.exists(images_path))
print("Labels folder exists:", os.path.exists(labels_path))
print("Classes file exists :", os.path.exists(classes_path))

# Count files
n_images = len(os.listdir(images_path)) if os.path.exists(images_path) else 0
n_labels = len(os.listdir(labels_path)) if os.path.exists(labels_path) else 0
print(f"\nFound {n_images} images and {n_labels} label files")

# Read classes
with open(classes_path) as f:
    classes = [line.strip() for line in f.readlines()]
print(f"\nClasses ({len(classes)}):")
for i, c in enumerate(classes):
    print(f"  {i}: {c}")

# ─────────────────────────────────────────────────────────────────
# CELL 4 — GAN: Augment your dataset (200 → ~1000 images)
# ─────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np
import shutil
import random
from tqdm import tqdm

# ── GAN Architecture ──────────────────────────────────────────────

class Generator(nn.Module):
    """
    DCGAN Generator
    Takes random noise vector → generates synthetic image patch
    """
    def __init__(self, nz=100, ngf=64, nc=3):
        super().__init__()
        self.main = nn.Sequential(
            # Input: nz × 1 × 1
            nn.ConvTranspose2d(nz, ngf*8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf*8),
            nn.ReLU(True),
            # → ngf*8 × 4 × 4
            nn.ConvTranspose2d(ngf*8, ngf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*4),
            nn.ReLU(True),
            # → ngf*4 × 8 × 8
            nn.ConvTranspose2d(ngf*4, ngf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*2),
            nn.ReLU(True),
            # → ngf*2 × 16 × 16
            nn.ConvTranspose2d(ngf*2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            # → ngf × 32 × 32
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh()
            # → nc × 64 × 64
        )

    def forward(self, x):
        return self.main(x)


class Discriminator(nn.Module):
    """
    DCGAN Discriminator
    Takes image → outputs probability of being real
    """
    def __init__(self, nc=3, ndf=64):
        super().__init__()
        self.main = nn.Sequential(
            # Input: nc × 64 × 64
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # → ndf × 32 × 32
            nn.Conv2d(ndf, ndf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*2),
            nn.LeakyReLU(0.2, inplace=True),
            # → ndf*2 × 16 × 16
            nn.Conv2d(ndf*2, ndf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*4),
            nn.LeakyReLU(0.2, inplace=True),
            # → ndf*4 × 8 × 8
            nn.Conv2d(ndf*4, ndf*8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*8),
            nn.LeakyReLU(0.2, inplace=True),
            # → ndf*8 × 4 × 4
            nn.Conv2d(ndf*8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
            # → 1 × 1 × 1
        )

    def forward(self, x):
        return self.main(x).view(-1, 1).squeeze(1)


def weights_init(m):
    """Initialize weights from paper (mean=0, std=0.02)"""
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


# ── Dataset for GAN training ──────────────────────────────────────

class CandyDataset(Dataset):
    def __init__(self, images_dir, image_size=64):
        self.images_dir = images_dir
        self.transform  = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
        ])
        self.image_files = [
            f for f in os.listdir(images_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ]
        print(f"  Dataset: {len(self.image_files)} images found")

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.images_dir, self.image_files[idx])
        img = Image.open(img_path).convert('RGB')
        return self.transform(img)


# ── GAN Training ─────────────────────────────────────────────────

def train_gan(
    images_dir,
    output_dir,
    n_epochs   = 100,     # increase to 200 for better quality
    batch_size = 32,
    nz         = 100,     # latent vector size
    lr         = 0.0002,
    beta1      = 0.5,
    n_generate = 800,     # how many fake images to generate
    image_size = 64,
    device     = None,
):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n  Training GAN on {device}")

    os.makedirs(output_dir, exist_ok=True)

    # Dataset & Loader
    dataset    = CandyDataset(images_dir, image_size)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2)

    # Models
    G = Generator(nz=nz).to(device)
    D = Discriminator().to(device)
    G.apply(weights_init)
    D.apply(weights_init)

    # Loss & Optimizers (paper: Adam β1=0.3, β2=0.978, lr=0.0011)
    criterion = nn.BCELoss()
    optG = optim.Adam(G.parameters(), lr=lr, betas=(beta1, 0.999))
    optD = optim.Adam(D.parameters(), lr=lr, betas=(beta1, 0.999))

    fixed_noise = torch.randn(16, nz, 1, 1, device=device)

    print(f"  Training for {n_epochs} epochs...")
    G_losses, D_losses = [], []

    for epoch in range(n_epochs):
        for i, real_imgs in enumerate(dataloader):
            real_imgs = real_imgs.to(device)
            b_size    = real_imgs.size(0)

            # ── Train Discriminator ──
            D.zero_grad()
            label_real = torch.ones(b_size, device=device)
            label_fake = torch.zeros(b_size, device=device)

            # Real images
            output   = D(real_imgs)
            loss_D_real = criterion(output, label_real)

            # Fake images
            noise    = torch.randn(b_size, nz, 1, 1, device=device)
            fake     = G(noise)
            output   = D(fake.detach())
            loss_D_fake = criterion(output, label_fake)

            loss_D = loss_D_real + loss_D_fake
            loss_D.backward()
            optD.step()

            # ── Train Generator ──
            G.zero_grad()
            output   = D(fake)
            loss_G   = criterion(output, label_real)   # want D to think fake = real
            loss_G.backward()
            optG.step()

        G_losses.append(loss_G.item())
        D_losses.append(loss_D.item())

        if (epoch + 1) % 10 == 0:
            print(f"  Epoch [{epoch+1:>3}/{n_epochs}]  "
                  f"Loss_D: {loss_D.item():.4f}  Loss_G: {loss_G.item():.4f}")

    # ── Generate synthetic images ─────────────────────────────────
    print(f"\n  Generating {n_generate} synthetic images...")
    G.eval()
    generated_count = 0

    with torch.no_grad():
        while generated_count < n_generate:
            batch = min(64, n_generate - generated_count)
            noise = torch.randn(batch, nz, 1, 1, device=device)
            fake  = G(noise)
            fake  = (fake * 0.5 + 0.5).clamp(0, 1)   # denormalize

            for j in range(batch):
                img_tensor = fake[j].cpu().permute(1, 2, 0).numpy()
                img_pil    = Image.fromarray((img_tensor * 255).astype(np.uint8))
                img_pil    = img_pil.resize((416, 416))   # resize for YOLO
                img_pil.save(os.path.join(output_dir, f"gan_{generated_count:04d}.jpg"))
                generated_count += 1

    print(f"  Saved {generated_count} GAN images to {output_dir}")
    return G, D, G_losses, D_losses


# ── Run GAN training ─────────────────────────────────────────────
GAN_OUTPUT_DIR = "/kaggle/working/gan_generated"

print("=" * 60)
print("STEP 1: TRAINING GAN FOR DATA AUGMENTATION")
print("=" * 60)

G, D, g_losses, d_losses = train_gan(
    images_dir  = images_path,
    output_dir  = GAN_OUTPUT_DIR,
    n_epochs    = 100,
    n_generate  = 800,
)

print("\nGAN training complete!")

# ─────────────────────────────────────────────────────────────────
# CELL 5 — Prepare combined dataset for YOLOv7
# ─────────────────────────────────────────────────────────────────
import shutil
import random
import os
from pathlib import Path

YOLO_DATASET_DIR = "/kaggle/working/candy_yolo"

# Create YOLOv7 folder structure
for split in ["train", "val"]:
    os.makedirs(f"{YOLO_DATASET_DIR}/images/{split}", exist_ok=True)
    os.makedirs(f"{YOLO_DATASET_DIR}/labels/{split}",  exist_ok=True)

print("=" * 60)
print("STEP 2: PREPARING YOLO DATASET")
print("=" * 60)

# ── Collect all real images ───────────────────────────────────────
all_images = sorted([
    f for f in os.listdir(images_path)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
])
print(f"\nReal images     : {len(all_images)}")

# ── Split real images 80% train / 20% val ────────────────────────
random.seed(42)
random.shuffle(all_images)
split_idx   = int(len(all_images) * 0.8)
train_imgs  = all_images[:split_idx]
val_imgs    = all_images[split_idx:]
print(f"Train split     : {len(train_imgs)} images")
print(f"Val split       : {len(val_imgs)} images")

# ── Copy real images + labels ─────────────────────────────────────
def copy_image_and_label(img_name, src_img_dir, src_lbl_dir, dst_split):
    stem      = Path(img_name).stem
    src_img   = os.path.join(src_img_dir, img_name)
    src_lbl   = os.path.join(src_lbl_dir, stem + ".txt")
    dst_img   = f"{YOLO_DATASET_DIR}/images/{dst_split}/{img_name}"
    dst_lbl   = f"{YOLO_DATASET_DIR}/labels/{dst_split}/{stem}.txt"

    shutil.copy2(src_img, dst_img)
    if os.path.exists(src_lbl):
        shutil.copy2(src_lbl, dst_lbl)
    else:
        # Create empty label file if missing
        open(dst_lbl, 'w').close()

for img in train_imgs:
    copy_image_and_label(img, images_path, labels_path, "train")
for img in val_imgs:
    copy_image_and_label(img, images_path, labels_path, "val")

# ── Add GAN images to TRAIN only ─────────────────────────────────
# GAN images have no labels → we assign them as background (empty label)
# This helps reduce false positives
gan_images = sorted([
    f for f in os.listdir(GAN_OUTPUT_DIR)
    if f.endswith('.jpg')
])[:400]   # use 400 GAN images for training

print(f"\nGAN images added to train: {len(gan_images)}")

for img_name in gan_images:
    src = os.path.join(GAN_OUTPUT_DIR, img_name)
    dst = f"{YOLO_DATASET_DIR}/images/train/{img_name}"
    shutil.copy2(src, dst)
    # Empty label = background image (helps reduce false positives)
    open(f"{YOLO_DATASET_DIR}/labels/train/{Path(img_name).stem}.txt", 'w').close()

# ── Count final dataset ───────────────────────────────────────────
n_train = len(os.listdir(f"{YOLO_DATASET_DIR}/images/train"))
n_val   = len(os.listdir(f"{YOLO_DATASET_DIR}/images/val"))
print(f"\nFinal dataset:")
print(f"  Train : {n_train} images")
print(f"  Val   : {n_val} images")
print(f"  Total : {n_train + n_val} images")

# ─────────────────────────────────────────────────────────────────
# CELL 6 — Create dataset YAML config for YOLOv7
# ─────────────────────────────────────────────────────────────────
import yaml

# Read classes
with open(classes_path) as f:
    classes = [line.strip() for line in f.readlines() if line.strip()]

dataset_config = {
    "path"  : YOLO_DATASET_DIR,
    "train" : "images/train",
    "val"   : "images/val",
    "nc"    : len(classes),
    "names" : classes,
}

yaml_path = "/kaggle/working/candy.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(dataset_config, f, default_flow_style=False)

print("Dataset config (candy.yaml):")
print("-" * 40)
with open(yaml_path) as f:
    print(f.read())

# ─────────────────────────────────────────────────────────────────
# CELL 7 — Download YOLOv7 pretrained weights
# ─────────────────────────────────────────────────────────────────
import os

os.chdir("/kaggle/working/yolov7")

# Download pretrained weights (trained on COCO)
# We fine-tune FROM these weights → much faster + better than from scratch
os.system("wget -q https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7.pt")

print("Pretrained weights downloaded!")
print("File size:", os.path.getsize("yolov7.pt") // (1024*1024), "MB")

# ─────────────────────────────────────────────────────────────────
# CELL 8 — Train YOLOv7
# ─────────────────────────────────────────────────────────────────
import os

os.chdir("/kaggle/working/yolov7")

# ── Training hyperparameters (optimized from paper Table 1) ──────
EPOCHS     = 100      # increase to 200 for best results
BATCH_SIZE = 16       # reduce to 8 if GPU OOM error
IMG_SIZE   = 640      # YOLOv7 default input size
WORKERS    = 4
NAME       = "candy_yolov7"

print("=" * 60)
print("STEP 3: TRAINING YOLOv7")
print(f"  Epochs     : {EPOCHS}")
print(f"  Batch size : {BATCH_SIZE}")
print(f"  Image size : {IMG_SIZE}")
print(f"  Classes    : {len(classes)}")
print("=" * 60)
print("\nThis will take ~1-2 hours on GPU T4...")
print("Watch the mAP and loss values below:\n")

train_cmd = f"""python train.py \
    --weights yolov7.pt \
    --cfg cfg/training/yolov7.yaml \
    --data {yaml_path} \
    --epochs {EPOCHS} \
    --batch-size {BATCH_SIZE} \
    --img {IMG_SIZE} \
    --workers {WORKERS} \
    --name {NAME} \
    --hyp data/hyp.scratch.p5.yaml \
    --project /kaggle/working/runs/train \
    --exist-ok \
    --cache"""

os.system(train_cmd)

# ─────────────────────────────────────────────────────────────────
# CELL 9 — Check training results
# ─────────────────────────────────────────────────────────────────
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

results_dir = f"/kaggle/working/runs/train/{NAME}"

# Check best weights exist
best_weights = os.path.join(results_dir, "weights", "best.pt")
last_weights = os.path.join(results_dir, "weights", "last.pt")

print("Training Results:")
print(f"  Best weights : {best_weights}")
print(f"  Exists       : {os.path.exists(best_weights)}")

if os.path.exists(best_weights):
    size_mb = os.path.getsize(best_weights) // (1024*1024)
    print(f"  Size         : {size_mb} MB")

# Plot training curves
results_csv = os.path.join(results_dir, "results.csv")
if os.path.exists(results_csv):
    import pandas as pd
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle("YOLOv7 Training Results — Candy Dataset", fontsize=14)

    plots = [
        ("train/box_loss",  "Box Loss (Train)",     axes[0,0]),
        ("train/obj_loss",  "Obj Loss (Train)",     axes[0,1]),
        ("train/cls_loss",  "Class Loss (Train)",   axes[0,2]),
        ("metrics/mAP_0.5", "mAP@0.5",              axes[1,0]),
        ("val/box_loss",    "Box Loss (Val)",        axes[1,1]),
        ("val/cls_loss",    "Class Loss (Val)",      axes[1,2]),
    ]

    for col, title, ax in plots:
        if col in df.columns:
            ax.plot(df[col].values)
            ax.set_title(title)
            ax.set_xlabel("Epoch")
            ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("/kaggle/working/training_curves.png", dpi=150)
    plt.show()
    print("\nTraining curves saved!")

    # Print final metrics
    last_row = df.iloc[-1]
    print(f"\nFinal Metrics (Epoch {len(df)}):")
    for col in ["metrics/mAP_0.5", "metrics/mAP_0.5:0.95",
                "metrics/precision", "metrics/recall"]:
        if col in df.columns:
            print(f"  {col.split('/')[-1]:>20} : {last_row[col]:.4f}")

# ─────────────────────────────────────────────────────────────────
# CELL 10 — Run validation & per-class accuracy
# ─────────────────────────────────────────────────────────────────
import os

os.chdir("/kaggle/working/yolov7")

best_weights = f"/kaggle/working/runs/train/{NAME}/weights/best.pt"

val_cmd = f"""python val.py \
    --weights {best_weights} \
    --data {yaml_path} \
    --img {IMG_SIZE} \
    --batch-size 16 \
    --conf-thres 0.25 \
    --iou-thres 0.45 \
    --task val \
    --name candy_val \
    --project /kaggle/working/runs/val \
    --exist-ok \
    --verbose"""

print("=" * 60)
print("STEP 4: VALIDATION")
print("=" * 60)
os.system(val_cmd)

# ─────────────────────────────────────────────────────────────────
# CELL 11 — Test inference on sample images
# ─────────────────────────────────────────────────────────────────
import os

os.chdir("/kaggle/working/yolov7")

# Pick a few val images to test on
val_images_dir = f"{YOLO_DATASET_DIR}/images/val"
sample_imgs    = os.listdir(val_images_dir)[:5]
sample_str     = " ".join([os.path.join(val_images_dir, i) for i in sample_imgs])

detect_cmd = f"""python detect.py \
    --weights {best_weights} \
    --source {val_images_dir} \
    --img-size {IMG_SIZE} \
    --conf-thres 0.25 \
    --iou-thres 0.45 \
    --name candy_detect \
    --project /kaggle/working/runs/detect \
    --exist-ok \
    --save-txt \
    --save-conf"""

print("=" * 60)
print("STEP 5: SAMPLE INFERENCE")
print("=" * 60)
os.system(detect_cmd)

# Display results
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

detect_dir = "/kaggle/working/runs/detect/candy_detect"
if os.path.exists(detect_dir):
    result_imgs = [
        f for f in os.listdir(detect_dir)
        if f.endswith(('.jpg', '.png'))
    ][:6]

    if result_imgs:
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        fig.suptitle("YOLOv7 Detection Results — Candy Dataset", fontsize=14)
        axes = axes.flatten()

        for i, img_name in enumerate(result_imgs):
            img = mpimg.imread(os.path.join(detect_dir, img_name))
            axes[i].imshow(img)
            axes[i].set_title(img_name, fontsize=9)
            axes[i].axis("off")

        for j in range(len(result_imgs), 6):
            axes[j].axis("off")

        plt.tight_layout()
        plt.savefig("/kaggle/working/detection_results.png", dpi=150)
        plt.show()
        print("Detection results displayed!")

# ─────────────────────────────────────────────────────────────────
# CELL 12 — Save everything for download
# ─────────────────────────────────────────────────────────────────
import shutil
import os

OUTPUT_DIR = "/kaggle/working/final_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Copy best weights
shutil.copy2(
    f"/kaggle/working/runs/train/{NAME}/weights/best.pt",
    f"{OUTPUT_DIR}/candy_best.pt"
)

# Copy last weights
shutil.copy2(
    f"/kaggle/working/runs/train/{NAME}/weights/last.pt",
    f"{OUTPUT_DIR}/candy_last.pt"
)

# Copy dataset yaml
shutil.copy2(yaml_path, f"{OUTPUT_DIR}/candy.yaml")

# Copy training curves
if os.path.exists("/kaggle/working/training_curves.png"):
    shutil.copy2(
        "/kaggle/working/training_curves.png",
        f"{OUTPUT_DIR}/training_curves.png"
    )

# Copy detection results
if os.path.exists("/kaggle/working/detection_results.png"):
    shutil.copy2(
        "/kaggle/working/detection_results.png",
        f"{OUTPUT_DIR}/detection_results.png"
    )

# Save classes.txt
shutil.copy2(classes_path, f"{OUTPUT_DIR}/classes.txt")

# List all outputs
print("=" * 60)
print("FILES READY FOR DOWNLOAD:")
print("=" * 60)
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f:<35} {size//1024:>8} KB")

print(f"\nAll files saved to: {OUTPUT_DIR}")
print("\nTo download: click the file in Kaggle Output panel →")
print("  candy_best.pt  ← USE THIS for inference on your laptop")
print("  candy.yaml     ← needed alongside the weights")
print("  classes.txt    ← class names")